In [ ]:
!pip install rdkit -q

# ==========================================================
# LARGE-SCALE VIRTUAL SCREENING OF NATURAL PRODUCTS
# ==========================================================
# This workflow screens compounds from the COCONUT/NPAtlas
# databases using three independently trained Extra Trees
# classifiers developed for:
#   1. Escherichia coli
#   2. Pseudomonas aeruginosa
#   3. Staphylococcus aureus

In [ ]:
import pandas as pd
import numpy as np
import joblib

from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold

In [ ]:
coconut = pd.read_csv("coconut.csv")

print(coconut.shape)
print(coconut.columns)
coconut.head()

/tmp/ipykernel_46227/3026082284.py:1: DtypeWarning: Columns (39) have mixed types. Specify dtype option on import or set low_memory=False.
  coconut = pd.read_csv("coconut.csv")


(738827, 40)
Index(['id', 'identifier', 'canonical_smiles', 'standard_inchi',
       'standard_inchi_key', 'name', 'iupac_name', 'annotation_level',
       'total_atom_count', 'heavy_atom_count', 'molecular_weight',
       'exact_molecular_weight', 'molecular_formula', 'alogp',
       'topological_polar_surface_area', 'rotatable_bond_count',
       'hydrogen_bond_acceptors', 'hydrogen_bond_donors',
       'hydrogen_bond_acceptors_lipinski', 'hydrogen_bond_donors_lipinski',
       'lipinski_rule_of_five_violations', 'aromatic_rings_count',
       'qed_drug_likeliness', 'formal_charge', 'fractioncsp3',
       'number_of_minimal_rings', 'van_der_walls_volume', 'contains_sugar',
       'contains_ring_sugars', 'contains_linear_sugars', 'murcko_framework',
       'np_likeness', 'chemical_class', 'chemical_sub_class',
       'chemical_super_class', 'direct_parent_classification',
       'np_classifier_pathway', 'np_classifier_superclass',
       'np_classifier_class', 'np_classifier_is_glycos

,id,identifier,canonical_smiles,standard_inchi,standard_inchi_key,name,iupac_name,annotation_level,total_atom_count,heavy_atom_count,...,murcko_framework,np_likeness,chemical_class,chemical_sub_class,chemical_super_class,direct_parent_classification,np_classifier_pathway,np_classifier_superclass,np_classifier_class,np_classifier_is_glycoside
0,925233,CNP0018900.0,COC1=CC(O)=C2C(O)=C3C(=O)CCC(=O)C3=C(C3=C(OC)C...,InChI=1S/C30H20O11/c1-40-11-7-12-20(18(35)8-11...,MKSHWBFPGCUPAF-UHFFFAOYSA-N,NaN,"2-(5,10-dihydroxy-7-methoxy-1,4-dioxo-2,3-dihy...",1,61,41,...,c1ccc2c(c1)cc3c(c2-c4ccc5c(c4)Cc6ccccc6C5)CCCC3,1.47,Arylnaphthalene lignans,Arylnaphthalene lignans,"Lignans, neolignans and related compounds",Arylnaphthalene lignans,Polyketides,Polycyclic aromatic polyketides,Anthraquinones and anthrones,False
1,925367,CNP0392312.0,COC1=CC(O)=C2C(O)=C3C(=O)CC(O)(OC)CC3=CC2=C1,"InChI=1S/C16H16O6/c1-21-10-4-8-3-9-6-16(20,22-...",VLKSTOXYUPUMDY-UHFFFAOYSA-N,NaN,"3,8,9-trihydroxy-3,6-dimethoxy-2,4-dihydroanth...",1,38,22,...,c1ccc2cc3c(cc2c1)CCCC3,1.58,Naphthalenes,Naphthols and derivatives,Benzenoids,Naphthols and derivatives,Polyketides,Naphthalenes,Naphthalenes and derivatives,False
2,925500,CNP0111030.14,C[C@H]1[C@H](C)CC[C@]2(C(=O)O)CC[C@]3(C(=O)O)C...,InChI=1S/C36H56O10/c1-18-9-14-35(30(41)42)15-1...,AXNXSFBKZQIMPF-LAKVJUMPSA-N,NaN,"(1~{S},2~{R},4~{a}~{S},6~{a}~{R},6~{a}~{R},6~{...",2,102,46,...,O1CCCCC1OC2CCC3C(CCC4C5C(=CCC43)C6CCCCC6CC5)C2,2.92,Steroids and steroid derivatives,Oxosteroids,Lipids and lipid-like molecules,Oxosteroids,Terpenoids,Triterpenoids,Ursane and Taraxastane triterpenoids,True
3,925502,CNP0570152.4,O=C(O)CC(=O)OC[C@H]1O[C@@H](OC2=CC(O)=CC3=[O+]...,InChI=1S/C42H40O26.ClH/c43-17-8-23-18(24(9-17)...,BUSHEYXQOAMBAI-PKDFCWOISA-N,Salviadelphin chloride,"3-[[(2~{R},3~{S},4~{R},5~{R},6~{S})-3-(2-carbo...",4,110,69,...,O1=c2cccc(OC3OCCCC3)c2=CC(OC4OC(COCC=Cc5ccccc5...,1.53,Flavonoids,Flavonoid glycosides,Phenylpropanoids and polyketides,Anthocyanidin-3-O-glycosides,Shikimates and Phenylpropanoids,Flavonoids,Anthocyanidins,True
4,925513,CNP0423858.1,COC1=CC(O)=C2C(=O)C(O[C@H]3O[C@@H](OC)[C@H](O[...,InChI=1S/C28H32O16/c1-9-17(32)19(34)21(36)26(4...,BDQAVVLZKLFTIK-KCYUVNIVSA-N,Rhamnetin-3-O-neohesperidoside,"3-[(2~{S},3~{S},4~{S},5~{R},6~{R})-3,4-dihydro...",4,76,44,...,O1c2ccccc2CC(OC3OCC(OC4OCCCC4)CC3)=C1c5ccccc5,1.95,Flavonoids,Flavonoid glycosides,Phenylpropanoids and polyketides,Flavonoid-3-O-glycosides,Shikimates and Phenylpropanoids,Flavonoids,Flavonols,True


In [ ]:
SMILES_COL = "smiles"
NAME_COL = "name"

In [ ]:
# SMILES_COL = "canonical_smiles"

In [ ]:
descriptor_cols = [
    "MW",
    "LogP",
    "TPSA",
    "HBD",
    "HBA",
    "RotBonds",
    "RingCount",
    "HeavyAtoms"
]

def calc_descriptors_and_fp(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(str(smiles))

    if mol is None:
        return None, None

    desc = {
        "MW": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
        "HBD": Descriptors.NumHDonors(mol),
        "HBA": Descriptors.NumHAcceptors(mol),
        "RotBonds": Descriptors.NumRotatableBonds(mol),
        "RingCount": Descriptors.RingCount(mol),
        "HeavyAtoms": Descriptors.HeavyAtomCount(mol)
    }

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius,
        nBits=n_bits
    )

    arr = np.zeros((n_bits,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)

    return desc, arr

In [ ]:
import pandas as pd
import numpy as np
import pickle
import gc
from tqdm import tqdm

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")
tqdm.pandas()

In [ ]:
# =========================
# FILE PATHS
# =========================

INPUT_FILE = "coconut.csv"

E_COLI_MODEL_FILE = "E_coli_ExtraTrees_model.pkl"
PA_MODEL_FILE = "P_aeruginosa_ExtraTrees_model.pkl"
SA_MODEL_FILE = "S_aureus_ExtraTrees_model.pkl"

ALL_RESULTS_FILE = "coconut_primary_screen_all_predictions.csv"
HITS_FILE = "coconut_primary_screen_broad_spectrum_hits.csv"

In [ ]:
# =========================
# COLUMN NAMES
# Change these if needed
# =========================

ID_COL = "identifier"
NAME_COL = "name"
SMILES_COL = "canonical_smiles"

CHUNK_SIZE = 50000
PROBABILITY_CUTOFF = 0.70

FP_RADIUS = 2
FP_BITS = 2048

In [ ]:
import joblib

# =========================
# LOAD MODELS
# =========================

with open(E_COLI_MODEL_FILE, "rb") as f:
    ecoli_model = joblib.load(f)

with open(PA_MODEL_FILE, "rb") as f:
    pa_model = joblib.load(f)

with open(SA_MODEL_FILE, "rb") as f:
    sa_model = joblib.load(f)

print("Models loaded successfully.")

Models loaded successfully.


In [ ]:
# =========================
# FINGERPRINT FUNCTION
# =========================

def smiles_to_morgan_fp(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))

        if mol is None:
            return None

        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius=FP_RADIUS,
            nBits=FP_BITS
        )

        arr = np.zeros((FP_BITS,), dtype=np.float32)
        AllChem.DataStructs.ConvertToNumpyArray(fp, arr)

        return arr

    except Exception:
        return None

In [ ]:
import pandas as pd
import numpy as np
import joblib
import gc
from tqdm import tqdm

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, DataStructs
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")
tqdm.pandas()

In [ ]:
# =========================
# FILES
# =========================

INPUT_FILE = "coconut.csv"

E_COLI_MODEL_FILE = "E_coli_ExtraTrees_model.pkl"
PA_MODEL_FILE = "P_aeruginosa_ExtraTrees_model.pkl"
SA_MODEL_FILE = "S_aureus_ExtraTrees_model.pkl"

ALL_RESULTS_FILE = "coconut_primary_screen_all_predictions_corrected.csv"
HITS_FILE = "coconut_primary_screen_broad_spectrum_hits_corrected.csv"

In [ ]:
# =========================
# SETTINGS
# =========================

ID_COL = "identifier"
NAME_COL = "name"
SMILES_COL = "canonical_smiles"

CHUNK_SIZE = 50000
PROBABILITY_CUTOFF = 0.70

FP_RADIUS = 2
FP_BITS = 2048

In [ ]:
# =========================
# LOAD MODELS
# =========================

ecoli_model = joblib.load(E_COLI_MODEL_FILE)
pa_model = joblib.load(PA_MODEL_FILE)
sa_model = joblib.load(SA_MODEL_FILE)

print("Models loaded successfully.")
print("E. coli features:", ecoli_model.n_features_in_)
print("P. aeruginosa features:", pa_model.n_features_in_)
print("S. aureus features:", sa_model.n_features_in_)

Models loaded successfully.
E. coli features: 2057
P. aeruginosa features: 2057
S. aureus features: 2057


In [ ]:
# =========================
# FEATURE FUNCTIONS
# =========================

def smiles_to_morgan_fp(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None

        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius=FP_RADIUS,
            nBits=FP_BITS
        )

        arr = np.zeros((FP_BITS,), dtype=np.float32)
        DataStructs.ConvertToNumpyArray(fp, arr)

        return arr

    except Exception:
        return None


def calc_additional_descriptors(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None

        desc = np.array([
            Descriptors.MolWt(mol),
            Descriptors.MolLogP(mol),
            Descriptors.TPSA(mol),
            Descriptors.NumHDonors(mol),
            Descriptors.NumHAcceptors(mol),
            Descriptors.NumRotatableBonds(mol),
            Descriptors.RingCount(mol),
            Descriptors.HeavyAtomCount(mol),
            Descriptors.FractionCSP3(mol)
        ], dtype=np.float32)

        return desc

    except Exception:
        return None

In [ ]:
# =========================
# PRIMARY SCREENING LOOP
# =========================

first_all = True
first_hits = True

total_processed = 0
total_valid = 0
total_hits = 0

for chunk_number, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):

    print(f"\nProcessing chunk {chunk_number + 1}")

    raw_chunk_size = chunk.shape[0]

    chunk = chunk[[ID_COL, NAME_COL, SMILES_COL]].copy()

    chunk.columns = [
        "compound_id",
        "compound_name",
        "canonical_smiles"
    ]

    chunk = chunk.dropna(subset=["canonical_smiles"]).copy()

    chunk["fingerprint"] = chunk["canonical_smiles"].progress_apply(
        smiles_to_morgan_fp
    )

    chunk["descriptors"] = chunk["canonical_smiles"].progress_apply(
        calc_additional_descriptors
    )

    valid_chunk = chunk[
        chunk["fingerprint"].notna() &
        chunk["descriptors"].notna()
    ].copy()

    if valid_chunk.empty:
        total_processed += raw_chunk_size
        print("No valid compounds in this chunk.")
        continue

    combined_features = [
        np.concatenate([fp, desc])
        for fp, desc in zip(
            valid_chunk["fingerprint"],
            valid_chunk["descriptors"]
        )
    ]

    X_screen = np.vstack(combined_features).astype(np.float32)

    if X_screen.shape[1] != ecoli_model.n_features_in_:
        raise ValueError(
            f"Feature mismatch: X_screen has {X_screen.shape[1]}, "
            f"but model expects {ecoli_model.n_features_in_}"
        )

    valid_chunk["ecoli_probability"] = ecoli_model.predict_proba(X_screen)[:, 1]
    valid_chunk["paeruginosa_probability"] = pa_model.predict_proba(X_screen)[:, 1]
    valid_chunk["saureus_probability"] = sa_model.predict_proba(X_screen)[:, 1]

    valid_chunk["combined_antibacterial_score"] = (
        valid_chunk["ecoli_probability"] +
        valid_chunk["paeruginosa_probability"] +
        valid_chunk["saureus_probability"]
    ) / 3

    valid_chunk = valid_chunk.drop(columns=["fingerprint", "descriptors"])

    valid_chunk.to_csv(
        ALL_RESULTS_FILE,
        mode="w" if first_all else "a",
        header=first_all,
        index=False
    )
    first_all = False

    hits = valid_chunk[
        (valid_chunk["ecoli_probability"] >= PROBABILITY_CUTOFF) &
        (valid_chunk["paeruginosa_probability"] >= PROBABILITY_CUTOFF) &
        (valid_chunk["saureus_probability"] >= PROBABILITY_CUTOFF)
    ].copy()

    hits = hits.sort_values(
        "combined_antibacterial_score",
        ascending=False
    )

    if not hits.empty:
        hits.to_csv(
            HITS_FILE,
            mode="w" if first_hits else "a",
            header=first_hits,
            index=False
        )
        first_hits = False

    total_processed += raw_chunk_size
    total_valid += valid_chunk.shape[0]
    total_hits += hits.shape[0]

    print("Processed:", total_processed)
    print("Valid:", total_valid)
    print("Broad-spectrum hits so far:", total_hits)

    del chunk, valid_chunk, X_screen, hits, combined_features
    gc.collect()

print("\nScreening completed.")
print("Total processed:", total_processed)
print("Total valid:", total_valid)
print("Total broad-spectrum antibacterial hits:", total_hits)

/tmp/ipykernel_46227/2735418505.py:12: DtypeWarning: Columns (27,39) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):



Processing chunk 1


100%|██████████| 50000/50000 [01:17<00:00, 642.16it/s]


Processed: 50000
Valid: 50000
Broad-spectrum hits so far: 1495


/tmp/ipykernel_46227/2735418505.py:12: DtypeWarning: Columns (39) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):


...


Processing chunk 14


100%|██████████| 50000/50000 [01:18<00:00, 637.41it/s]


Processed: 700000
Valid: 699996
Broad-spectrum hits so far: 20296

Processing chunk 15


100%|██████████| 38827/38827 [01:00<00:00, 645.01it/s]


Processed: 738827
Valid: 738823
Broad-spectrum hits so far: 21390

Screening completed.
Total processed: 738827
Total valid: 738823
Total broad-spectrum antibacterial hits: 21390


# ==========================================================
# BROAD-SPECTRUM HIT IDENTIFICATION
# ==========================================================
# Filter primary screening results to retain only compounds
# predicted to be active against all three target pathogens.
#
# Selection criteria:
#   E. coli probability          ≥ 0.80
#   P. aeruginosa probability    ≥ 0.80
#   S. aureus probability        ≥ 0.80

In [ ]:
import pandas as pd

file_path = "coconut_primary_screen_broad_spectrum_hits_corrected.csv"

df = pd.read_csv(file_path)

screened = df[
    (df["ecoli_probability"] >= 0.80) &
    (df["paeruginosa_probability"] >= 0.80) &
    (df["saureus_probability"] >= 0.80)
]

screened.to_csv("screened_hits_0.80.csv", index=False)

print(screened.shape)
print(screened.head())

(1578, 7)
    compound_id  compound_name  \
0  CNP0193434.5            NaN   
1  CNP0140462.2            NaN   
2  CNP0200858.1    Thienamycin   
3  CNP0316445.0    Aloe-emodin   
4  CNP0336464.0  sulfisoxazole   

                                    canonical_smiles  ecoli_probability  \
0  NCC1O[C@H](O[C@@H]2C(N)C[C@@H](N)C(O)[C@@H]2OC...                1.0   
1  NC[C@H]1O[C@H](O[C@H]2[C@H](O)[C@@H](O[C@H]3O[...                1.0   
2   C[C@@H](O)[C@H]1C(=O)N2C(C(=O)O)=C(SCCN)C[C@H]12                1.0   
3         O=C1C2=CC=CC(O)=C2C(=O)C2=C(O)C=C(CO)C=C12                1.0   
4              CC1=NOC(NS(=O)(=O)C2=CC=C(N)C=C2)=C1C                1.0   

   paeruginosa_probability  saureus_probability  combined_antibacterial_score  
0                      1.0                  1.0                           1.0  
1                      1.0                  1.0                           1.0  
2                      1.0                  1.0                           1.0  
3             

In [ ]:
import pandas as pd
import numpy as np
import joblib
import gc
from tqdm import tqdm

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, DataStructs
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")
tqdm.pandas()

In [ ]:
# =========================
# LOAD MODELS
# =========================

ecoli_model = joblib.load(E_COLI_MODEL_FILE)
pa_model = joblib.load(PA_MODEL_FILE)
sa_model = joblib.load(SA_MODEL_FILE)

print("Models loaded successfully.")
print("E. coli features:", ecoli_model.n_features_in_)
print("P. aeruginosa features:", pa_model.n_features_in_)
print("S. aureus features:", sa_model.n_features_in_)

Models loaded successfully.
E. coli features: 2057
P. aeruginosa features: 2057
S. aureus features: 2057


In [ ]:
# =========================
# LOAD EXTRATREES MODELS ONLY
# =========================

ecoli_model = joblib.load("E_coli_ExtraTrees_model.pkl")
pa_model = joblib.load("P_aeruginosa_ExtraTrees_model.pkl")
sa_model = joblib.load("S_aureus_ExtraTrees_model.pkl")

print("Models loaded successfully.")
print("E. coli features:", ecoli_model.n_features_in_)
print("P. aeruginosa features:", pa_model.n_features_in_)
print("S. aureus features:", sa_model.n_features_in_)

Models loaded successfully.
E. coli features: 2057
P. aeruginosa features: 2057
S. aureus features: 2057


In [ ]:
import pandas as pd
import numpy as np
import joblib
import gc
from tqdm import tqdm

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, DataStructs
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")
tqdm.pandas()

# ==========================================================
# NPATLAS EXTERNAL VALIDATION AND VIRTUAL SCREENING
# ==========================================================
# The finalized Extra Trees antibacterial activity models
# were applied to the complete NPAtlas database to identify
# potential broad-spectrum antibacterial natural products.

In [ ]:
# =========================
# NPAtlas FILES
# =========================

INPUT_FILE = "NPAtlas.csv"

ALL_RESULTS_FILE = "NPAtlas_all_predictions_corrected.csv"
HITS_FILE = "NPAtlas_broad_spectrum_hits_corrected.csv"

CHUNK_SIZE = 50000
PROBABILITY_CUTOFF = 0.80

FP_RADIUS = 2
FP_BITS = 2048

In [ ]:
# =========================
# CHECK NPAtlas COLUMNS
# =========================

cols = pd.read_csv(INPUT_FILE, nrows=1, encoding="latin1").columns
print(cols)

Index(['npaid', 'compound_id', 'compound_names', 'compound_molecular_formula',
       'compound_molecular_weight', 'compound_accurate_mass',
       'compound_m_plus_h', 'compound_m_plus_na', 'compound_inchi',
       'compound_inchikey', 'compound_smiles', 'compound_cluster_id',
       'compound_node_id', 'origin_type', 'genus', 'origin_species',
       'original_reference_author_list', 'original_reference_year',
       'original_reference_issue', 'original_reference_volume',
       'original_reference_pages', 'original_reference_doi',
       'original_reference_pmid', 'original_reference_title',
       'original_reference_type', 'original_journal_title',
       'reassignment_dois', 'synthesis_dois', 'mibig_ids', 'gnps_ids',
       'npmrd_id', 'npatlas_url'],
      dtype='object')


In [ ]:
ID_COL = "npaid"
NAME_COL = "compound_names"
SMILES_COL = "compound_smiles"

In [ ]:
# =========================
# LOAD EXTRATREES MODELS
# =========================

ecoli_model = joblib.load("E_coli_ExtraTrees_model.pkl")
pa_model = joblib.load("P_aeruginosa_ExtraTrees_model.pkl")
sa_model = joblib.load("S_aureus_ExtraTrees_model.pkl")

print("Models loaded.")
print(ecoli_model.n_features_in_)
print(pa_model.n_features_in_)
print(sa_model.n_features_in_)

Models loaded.
2057
2057
2057


In [ ]:
# =========================
# FEATURE FUNCTIONS
# =========================

def smiles_to_morgan_fp(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None

        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius=FP_RADIUS,
            nBits=FP_BITS
        )

        arr = np.zeros((FP_BITS,), dtype=np.float32)
        DataStructs.ConvertToNumpyArray(fp, arr)

        return arr

    except:
        return None


def calc_additional_descriptors(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None

        return np.array([
            Descriptors.MolWt(mol),
            Descriptors.MolLogP(mol),
            Descriptors.TPSA(mol),
            Descriptors.NumHDonors(mol),
            Descriptors.NumHAcceptors(mol),
            Descriptors.NumRotatableBonds(mol),
            Descriptors.RingCount(mol),
            Descriptors.HeavyAtomCount(mol),
            Descriptors.FractionCSP3(mol)
        ], dtype=np.float32)

    except:
        return None

In [ ]:
# =========================
# NPAtlas EXTERNAL VALIDATION LOOP
# =========================

first_all = True
first_hits = True

total_processed = 0
total_valid = 0
total_hits = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE, encoding="latin1", low_memory=False)
):

    print(f"\nProcessing NPAtlas chunk {chunk_number + 1}")

    raw_chunk_size = chunk.shape[0]

    chunk = chunk[[ID_COL, NAME_COL, SMILES_COL]].copy()

    chunk.columns = [
        "compound_id",
        "compound_name",
        "canonical_smiles"
    ]

    chunk = chunk.dropna(subset=["canonical_smiles"]).copy()

    chunk["fingerprint"] = chunk["canonical_smiles"].progress_apply(
        smiles_to_morgan_fp
    )

    chunk["descriptors"] = chunk["canonical_smiles"].progress_apply(
        calc_additional_descriptors
    )

    valid_chunk = chunk[
        chunk["fingerprint"].notna() &
        chunk["descriptors"].notna()
    ].copy()

    if valid_chunk.empty:
        total_processed += raw_chunk_size
        print("No valid compounds in this chunk.")
        continue

    combined_features = [
        np.concatenate([fp, desc])
        for fp, desc in zip(
            valid_chunk["fingerprint"],
            valid_chunk["descriptors"]
        )
    ]

    X_screen = np.vstack(combined_features).astype(np.float32)

    if X_screen.shape[1] != ecoli_model.n_features_in_:
        raise ValueError(
            f"Feature mismatch: X_screen has {X_screen.shape[1]}, "
            f"but model expects {ecoli_model.n_features_in_}"
        )

    valid_chunk["ecoli_probability"] = ecoli_model.predict_proba(X_screen)[:, 1]
    valid_chunk["paeruginosa_probability"] = pa_model.predict_proba(X_screen)[:, 1]
    valid_chunk["saureus_probability"] = sa_model.predict_proba(X_screen)[:, 1]

    valid_chunk["combined_antibacterial_score"] = (
        valid_chunk["ecoli_probability"] +
        valid_chunk["paeruginosa_probability"] +
        valid_chunk["saureus_probability"]
    ) / 3

    valid_chunk = valid_chunk.drop(columns=["fingerprint", "descriptors"])

    valid_chunk.to_csv(
        ALL_RESULTS_FILE,
        mode="w" if first_all else "a",
        header=first_all,
        index=False
    )
    first_all = False

    hits = valid_chunk[
        (valid_chunk["ecoli_probability"] >= PROBABILITY_CUTOFF) &
        (valid_chunk["paeruginosa_probability"] >= PROBABILITY_CUTOFF) &
        (valid_chunk["saureus_probability"] >= PROBABILITY_CUTOFF)
    ].copy()

    hits = hits.sort_values(
        "combined_antibacterial_score",
        ascending=False
    )

    if not hits.empty:
        hits.to_csv(
            HITS_FILE,
            mode="w" if first_hits else "a",
            header=first_hits,
            index=False
        )
        first_hits = False

    total_processed += raw_chunk_size
    total_valid += valid_chunk.shape[0]
    total_hits += hits.shape[0]

    print("Processed:", total_processed)
    print("Valid:", total_valid)
    print("NPAtlas broad-spectrum hits so far:", total_hits)

    del chunk, valid_chunk, X_screen, hits, combined_features
    gc.collect()

print("\nNPAtlas external validation completed.")
print("Total processed:", total_processed)
print("Total valid:", total_valid)
print("Total broad-spectrum antibacterial hits:", total_hits)


Processing NPAtlas chunk 1


100%|██████████| 36395/36395 [00:54<00:00, 672.67it/s]


Processed: 36395
Valid: 36394
NPAtlas broad-spectrum hits so far: 213

NPAtlas external validation completed.
Total processed: 36395
Total valid: 36394
Total broad-spectrum antibacterial hits: 213
